# 5. 레이블이 없는 데이터를 활용한 사전 훈련 (자기지도학습)

## 5.1. 텍스트 생성 모델 평가

### 5.1.1. GPT를 사용한 텍스트 생성

#### GPTModel 클래스 및 GPT_CONFIG_124M을 사용하여 GPT 모델 초기화

In [ ]:
import torch
import torch.nn as nn
from torch.nn import LayerNorm

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        # 원하는 출력 차원에 맞도록 투영 차원을 낮춤
        self.head_dim = d_out // num_heads

        # Query, Key, Value 선형 변환
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 여러 head의 출력을 다시 섞어주는 출력 projection
        # Linear층을 사용해 헤드의 출력을 결합
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        # causal mask
        # 위쪽 삼각행렬을 1로 만들어서 미래 토큰을 보지 못하게 함
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        # 텐서 크기: (b, num_tokens, d_out)
        # x: (batch_size, num_tokens, d_in)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads 차원을 추가함으로써 암묵적으로 행렬 분할
        # 그런 다음 마지막 차원을 num_heads에 맞춰 채움
        # keys, queries, values:
        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)


        # head 차원을 앞으로 이동
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # attention score 계산
        # queries: (b, num_heads, num_tokens, head_dim)
        # keys.transpose(2, 3): (b, num_heads, head_dim, num_tokens)
        # 결과: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # 현재 토큰 수에 맞게 mask 자르기
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 미래 토큰 위치를 -inf로 채움
        # 마스크를 사용해 어텐션 점수를 채움
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # scaled dot-product attention
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        # attention weight와 value 곱하기
        # attn_weights: (b, num_heads, num_tokens, num_tokens)
        # values:       (b, num_heads, num_tokens, head_dim)
        # 결과:          (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # 다시 head 차원을 뒤로 보냄
        # 텐서 크기: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # 여러 head를 하나로 결합
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        # 마지막 출력 projection (선형 투영)
        context_vec = self.out_proj(context_vec)

        return context_vec

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x # 어텐션 블록을 위한 숏컷 연결
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # 원본 입력을 더함

        shortcut = x # 피드포워드 신경망을 위한 숏컷 연결
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # 원본 입력을 더함

        return x

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        
        # 장치 설정을 통해 입력 데이터가 어디에 있는지에 따라 모델을 CPU나 GPU에서 훈련 가능
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
import torch
GPT_CONFIG_124M = {
    "vocab_size": 50257, # 토크나이저의 어휘 크기
    "context_length": 256, # 모델이 한 번에 처리할 수 있는 최대 토큰 수 (문맥 길이를 1,024에서 256으로 줄임)
    "emb_dim": 768, # 토큰 임베딩의 차원
    "n_heads": 12, # 멀티헤드 어텐션에서 헤드 수
    "n_layers": 12, # 트랜스포머 블록 수
    "drop_rate": 0.1, # 드롭아웃 확률
    "qkv_bias": False, # 쿼리, 키, 값에 대한 편향 사용 여부
} 
torch.manual_seed(123) # 재현성을 위해 시드 설정
model = GPTModel(GPT_CONFIG_124M) # 모델 초기화
model.eval() # 모델을 평가 모드로 전환하여 드롭아웃과 같은 훈련 전용 동작을 비활성화

#### GPTModel의 인스턴스 객체를 4장의 generate_text_simple 함수에 적용
#### + 2개의 유틸리티 함수 text_to_token_ids와 token_ids_to_text 만들기
    - 텍스트와 토큰 표현 사이의 변환 수행
1. 토크나이저가 입력 텍스트를 토큰 ID로 변환
2. 모델이 토큰 ID를 받아 로짓 생성 (로짓: 어휘사전 내 각 토큰에 대한 확률분포를 나타내는 벡터)
3. 로짓을 다시 토큰 ID로 바꿈
4. 토크나이저가 토큰 ID를 사람이 읽을 수 있는 텍스트로 디코딩하여 텍스트 입력 ~ 출력 사이클 완료

#### 텍스트를 토큰 ID로 변환하기 위한 유틸리티 함수

In [ ]:
def generate_text_simple(model, idx, # idx는 현재 문맥이 담긴 [batch_size, num_tokens] 크기의 인덱스 배열
                         max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        
        logits = logits[:, -1, :] # 마지막 타임 스텝만 사용하므로 [batch_size, vocab_size]가 [batch_size, num_tokens, vocab_size]에서 [batch_size, vocab_size]로 축소됨
        probas = torch.softmax(logits, dim=-1) # probas는 [batch_size, vocab_size] 크기의 확률 분포
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) # idx_next는 [batch_size, 1] 크기의 텐서로, 각 샘플에 대해 가장 높은 확률을 가진 다음 토큰의 인덱스
        idx = torch.cat((idx, idx_next), dim=1) # 선택한 인덱스를 현재 시퀀스에 추가하므로 idx의 크기는 [batch_size, num_tokens + 1]이 됨
    
    return idx

In [ ]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # unsqueeze(0)으로 배치 차원을 추가
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 배치 차원 제거
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

#### 5.1.2. 텍스트 생성 손실 (text generation loss) 계산

* 입력 텍스트 -> 출력 텍스트 생성의 전반적인 흐름
    - 텍스트 생성 과정: generate_text_simple 함수가 내부적으로 수행하는 작업
    - 그림에서는 7개의 토큰으로 구성된 작은 어휘사전 사용
    - GPTModel은 50,257개의 단어로 구성된 훨씬 큰 어휘사전 사용
    - 토큰 ID는 0~6이 아닌 0~50,256 사이의 범위를 가짐

#### 2개의 입력 샘플
    - "every effort moves"
    - "I really like"

In [ ]:
inputs = torch.tensor ([[16833, 3626, 6100],  # ["every effort moves", 
                        [40,    1107, 588]])  #  "I really like"]

#### 입력에 대응하는 targets -> 모델이 생성할 토큰 ID를 담고 있음
    - targets는 한 위치 앞으로 이동한 입력

In [ ]:
targets = torch.tensor([[3626, 6100, 345],   # [ "effort moves you
                        [1107, 588, 11311]]) #   "really like chocolate"]

#### 입력을 모델에 주입 + 각 3개의 토큰으로 구성된 입력 샘플 2개에 대한 로짓 벡터 계산
#### + softmax 함수를 적용해 로짓을 확률 점수(probas)로 변환

In [ ]:
with torch.no_grad(): # 아직 훈련하지 않기 때문에 그래디언트 추적 비활성화
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1) # 어휘사전의 각 토큰에 대한 확률

print(logits.shape)
print(probas.shape)    

* 2: 샘플 크기(배치 크기)
* 3: 각 입력(샘플)당 토큰 개수
* 50257: 임베딩 차원 (어휘사전 크기)

* softmax 함수로 로짓을 확률로 변환한 후에 generate_text_simple 함수는 확률 점수를 다시 텍스트로 변환
    - 3 ~ 5 단계

#### 확률 점수에 argmax 적용하여 토큰 ID 추출 -> 3, 4단계 완료

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True) # 가장 높은 확률을 가진 토큰 인덱스 선택
print("토큰 ID:\n", token_ids)

#### 토큰 ID를 텍스트로 변환

In [ ]:
print(f"첫 번째 샘플의 타겟: {token_ids_to_text(targets[0], tokenizer)}")
print(f"두 번째 샘플의 타겟: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

#### 모델 훈련의 목적: 정답 타겟 토큰 ID에 해당하는 인덱스의 softmax 확률을 증가시키는 것
    - 정답 인덱스 위치의 확률이 높을수록 더 좋은 것


#### 2개의 입력 텍스트에 대해 타겟 토큰에 해당하는 초기 소프트맥스 확률 점수 출력

* 첫 번째 샘플의 타겟 토큰에 대한 확률:
    - tensor([7.4514e-05, 3.1054e-05, 1.1567e-05])
* 두 번째 샘플의 타겟 토큰에 대한 확률:
    - tensor([1.0343e-05, 5.6737e-05, 4.7620e-06])

* **LLM 훈련 목표: 정답 토큰의 가능성 최대화하기**

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]] # 첫 번째 샘플의 타겟 토큰에 대한 확률
print("첫 번째 샘플의 타겟 토큰에 대한 확률:\n", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]] # 두 번째 샘플의 타겟 토큰에 대한 확률
print("두 번째 샘플의 타겟 토큰에 대한 확률:\n", target_probas_2)

#### 두 샘플의 확률 점수 target_probas_1 & target_probas_2에 대한 손실 계산

In [ ]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print("로그 확률:\n", log_probas)

In [ ]:
# 로그 확률 평균하여 하나의 점수로 만들기
avg_log_probas = torch.mean(log_probas)
print("평균 로그 확률:\n", avg_log_probas)

#### 훈련 과정에서 모델 가중치를 업데이트하여 평균 로그 확률이 가능한 0에 가깝게 만드는 것이 목표
    - 딥러닝에서는 일반적으로 평균 로그 확률을 0으로 만드는 것보다 음의 평균 로그 확률을 0으로 만듦

In [ ]:
neg_avg_log_probas = -avg_log_probas
print("음수 평균 로그 확률 (손실):\n", neg_avg_log_probas)

#### 로짓과 타겟 텐서 크기 확인

In [ ]:
print("로짓 크기:", logits.shape)   # [배치 크기, 토큰 개수, 어휘사전 크기]
print("타겟 크기:", targets.shape)  # [배치 크기, 토큰 개수]

#### pytorch의 cross_entropy loss function을 위해 처음 두 차원을 결합하여 두 텐서 펼치기

In [ ]:
logits_flat = logits.flatten(0, 1) # 확률 점수르 얻기 위해 소프트맥스 함수로 들어가기 전의 정규화되지 않은 모델 출력
targets_flat = targets.flatten() # 모델이 생성하길 기대하는 토큰 ID
print("펼친 로짓:", logits_flat.shape)
print("펼친 타겟:", targets_flat.shape)

#### pytorch의 cross_entropy 
- softmax 함수 적용 + 타겟 ID에 해당하는 확률 점수 선택 + 음의 평균 로그 확률 계산 모두 수행

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

### 5.1.3. 훈련 세트와 검증 세트의 손실 계산
    - 훈련 데이터셋 / 검증 데이터셋 준비

In [ ]:
file_path = "the-verdict.txt"
with open(file_path, "r", encoding = "utf-8") as file:
    text_data = file.read()

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("문자 수:", total_characters)
print("토큰 수: ", total_tokens)

#### 데이터셋 -> 훈련 + 검증
    - 2장의 data loader -> llm 훈련을 위한 배치 준비
    - 그림을 위해 max_length=6 사용
    - 실제 data loader에서는 llm이 훈련 과정에서 더 긴 텍스트를 볼 수 있도록 max_length를 256 토큰으로 지정
* train_ratio = 0.9

In [ ]:
train_ratio = 0.90
split_idx = int(train_ratio + len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader   

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride) :
        self.input_ids = []
        self.target_ids = []
        
        # 전체 텍스트를 토큰화함
        token_ids = tokenizer.encode(txt)
        
        # 슬라이딩 윈도우를 사용해 문서를 max_length 길이의 중첩된 시퀀스로 나눔
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # 데이터셋에 있는 전체 행 수를 반환하는 메서드
    def __len__(self):
        return len(self.input_ids)
    
    # 데이터셋에서 하나의 행을 반환하는 메서드
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                      stride=128, shuffle=True, drop_last=True,
                      num_workers=0):
    # 토크나이저 초기화
    tokenizer = tiktoken.get_encoding("gpt2")
    
    # GPTDatasetV1 데이터셋 인스턴스 생성
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        
        # drop_last=True로 설정하면 batch_size보다 작을 경우 
        # 훈련 손실이 갑자기 높아지는 것을 피하기 위해 마지막 배치를 삭제함
        drop_last=drop_last,
        
        # 전처리에 사용할 CPU 프로세서 개수
        num_workers=num_workers
    )

    return dataloader

In [ ]:
torch.manual_seed(123)

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

#### 아래 코드 설명
* 각 배치: 2개의 샘플
* 각 샘플: 256개 토큰
* 데이터의 10%만 검증 세트에 할당 -> 배치가 1개뿐 (샘플 2개)
* 입력 데이터(x)와 타겟 데이터(y)의 크기가 동일
    - 각 배치 크기 = 샘플 개수(2) x 토큰 개수 (256)

In [ ]:
print("훈련 데이터 로더:")
for x, y in train_loader: 
    print(x.shape, y.shape)
    
print("\n검증 데이터 로더:")
for x, y in val_loader:
    print(x.shape, y.shape)

#### train_loader, val_loader가 반환한 배치로 cross_entropy loss 계산 유틸리티 함수

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    # 데이터를 지정된 장치 (GPU)로 전송
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

#### 하나의 배치에 대한 손실을 계산하는 calc_loss_batch 함수 -> calc_loss_loader 구현
    - data loader가 반환하는 모든 배치에 대한 손실 계산
    - 훈련 손실, 검증 손실 계산

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)  # num_batches가 지정되지 않으면 모든 배치 순회
    else:   # num_batches가 데이터 로더에 있는 배치 개수보다 크면 배치 횟수를 데이터 로더에 있는 총 배치 개수로 맞춤
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()   # 각 배치의 손실을 더함
        else:
            break
    
    return total_loss / num_batches  # 모든 배치의 손실 평균

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # CUDA 지원 GPU가 있다면 코드를 수정하지 않고 GPU에서 LLM을 훈련 가능

with torch.no_grad():  # 훈련하는 것이 아니므로 효율성을 위해 그래디언트 추적을 끔
    # device 매개변수를 사용해 LLM 모델과 동일한 장치에 데이터를 로드
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)  

print("훈련 손실:", train_loss)
print("검증 손실:", val_loss)

#### 이제 훈련할 차례

## 5.2. LLM 훈련

* 전형적인 pytorch 신경망 훈련 workflow

#### LLM 사전 훈련을 위한 함수

* train_model_simple 함수는 아직 구현하지 않은 2개 함수 사용

1. evaluate_model (7단계)
    - 모델이 업데이트된 후 모델이 향상되었는지 평가할 수 있도록 훈련 세트 / 검증 세트 손실 출력
    - 모델을 평가 모드로 설정하여 그래디언트 추적과 드롭아웃을 비활성화
2. generate_and_print_sample
    - 모델이 훈련 과정에서 향상되었는지 추적하기 위해 편의상 만든 함수
    - 텍스트 조각(start_context)을 입력으로 받아 토큰 ID로 변환하고 LLM에게 주입하여 앞서 만든 generate_text_simple로 샘플 텍스트 생성

* evaluate_model 함수가 모델 훈련 진행 과정에 대해 수치로 추정값을 제공
* generate_and_print_sample 함수는 모델이 생성한 구체적인 텍스트 샘플 제공 (이를 통해 훈련하는 동안 모델 성능 평가)

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval() # 안정적이고 재현 가능한 결과를 위해 평가하는 동안 드롭아웃 비활성화
    with torch.no_grad(): # 계산 오버헤드를 줄이기 위해 훈련 과정에서 필요하지 않은 그래디언트 추적을 끔
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train() # 훈련 모드로 다시 전환
    return train_loss, val_loss

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " ")) # 간결한 출력 포맷
    model.train()

In [ ]:
def train_model_simple(model, train_loader, val_loader,
                      optimizer, device, num_epochs,
                      eval_freq, eval_iter, start_context, tokenizer):
    
    # 손실과 지금까지 처리한 토큰 수를 추적하기 위해 리스트 초기화
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    
    for epoch in range(num_epochs): # 메인 훈련 루프 시작
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # 이전 배치 반복에서 얻은 손실 그래디언트를 초기화
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            loss.backward() # 손실 그래디언트 계산
            optimizer.step() # 손실 그래디언트를 사용하여 모델의 가중치 업데이트
            tokens_seen += input_batch.numel()
            global_step += 1
            
            if global_step % eval_freq == 0:  # 추가적인 평가 단계
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch {epoch+1} (Step {global_step:06d}): "
                      f"훈련 손실 {train_loss:.3f}, "
                      f"검증 손실 {val_loss:.3f}")
        
        generate_and_print_sample( # 각 에포크 후에 샘플 텍스트 출력
            model, tokenizer, device, start_context
        )
    
    return train_losses, val_losses, track_tokens_seen

#### Adam W 옵티마이저와 train_model_sample 함수로 GPTModel 객체를 10 epoch 훈련

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), # parameters() 메서드는 모델에 있는 훈련 가능한 모든 가중치 파라미터 반환
    lr=0.0004, weight_decay=0.1
)
num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

#### 훈련 과정 중 출력

* 훈련 손실이 극적으로 줄었음
    - 모델의 언어 능력 향상
    - 처음에는 모델이 시작 문맥 뒤에 쉼표만 추가하거나 and 단어만 반복했으나 훈련 마지막에는 정확한 텍스트 생성
    - 검증 세트 손실도 훈련하는 동안 감소하지만, 훈련 세트 손실만큼 작아지지는 않음

#### 훈련 세트 손실 vs. 검증 세트 손실

In [ ]:
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import torch

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # 아래 x축: Epoch
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="--", label="Validation loss")

    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    # x축 범위 제한
    ax1.set_xlim(0, epochs_seen[-1])

    # 위쪽 x축: Tokens seen
    ax2 = ax1.twiny()
    ax2.set_xlim(tokens_seen[0], tokens_seen[-1])
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()
    plt.show()


epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))

plot_losses(
    epochs_tensor,
    tokens_seen,
    train_losses,
    val_losses
)

#### 훈련 손실과 검증 손실 차이의 발산
* 훈련 데이터에 과적합됨
* "The Verdict" 텍스트 파일에서 모델이 생성한 텍스트 "quite insensible to the irony"와 같은 구절을 찾을 수 있으므로 모델이 훈련 데이터를 그대로 기억하고 있음을 알 수 있음

* 매우 작은 훈련 세트 + 여러 epoch 훈련 -> 훈련 데이터 통암기 현상

* 일반적인 LLM 모델은 훨씬 큰 데이터셋에서 딱 한 번의 epoch동안 훈련함

## 5.3. 무작위성을 제어하기 위한 디코딩 전략
* 독창적인 텍스트 생성을 위한 텍스트 생성 전략 (= 디코딩 전략)

#### 작은 모델로 추론할 때는 GPU 필요 x
- 모델을 GPU에서 CPU로 다시 옮기기

In [ ]:
model.to('cpu')
model.eval()

#### generate_and_print_sample 함수의 generate_text_simple 함수 살펴보기
* GPTModel의 객체를 generate_text_simple 함수에 전달하여 한 번에 한 토큰씩 텍스트 생성
    - 이 토큰은 어휘사전에 있는 토큰 중 가장 높은 확류 점수를 가진 토큰임
    - Every effort moves you에서 generate_text_simple 함수를 여러 번 실행해도 같은 출력 생성

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 텍스트: \n", token_ids_to_text(token_ids, tokenizer))

### 5.3.1. 온도(temperature) 스케일링
* greedy decoding: torch.argmax를 사용하여 가장 높은 확률을 가진 토큰을 선택
* argmax -> 확률 분포로부터 샘플링하는 함수로 바꿈 (다양성이 더 높은 텍스트 생성)

In [ ]:
vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
}

# key와 value 바꿈
inverse_vocab = {v: k for k, v in vocab.items()}

#### LLM이 시작 문맥으로 "every efforts makes you"를 받고 다음과 같은 로짓을 생성했다고 가정

In [ ]:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

#### generate_text_simple -> 로짓을 softmax 함수를 통해 확률로 변환
* 그 후 argmax 함수를 통해 생성 토큰에 해당하는 토큰 ID를 얻음
* 이를 역어휘사전으로 다시 매핑

In [ ]:
probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()
print(inverse_vocab[next_token_id])

#### argmax 함수를 multinomial 함수로 바꾸기
* multinomial 함수는 확률 점수에 비례해서 다음 토큰 샘플링
* forward가 가장 가능성이 높은 토큰이라 대부분의 경우 multinomial 함수에 의해 선택

In [ ]:
probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

#### multinomial 함수로 1000번 샘플링

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item()
        for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")
        
print_sampled_tokens(probas)

#### 온도(temperature) 스케일링
* 로짓을 0보다 더 큰 수로 나누는 방법
* temperature > 1: 조금 더 균등한 토큰 확률 분포
    - 너무 텍스트 다양성을 높이면 말도 안 되는 텍스트를 만들기도 함
* temperature < 1: 조금 더 결정론적인 분포 (더 날카롭거나 뾰족한 분포)

In [ ]:
import matplotlib.pyplot as plt

def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

temperatures = [1.0, 0.1, 5.0] # [원본, 낮은 온도, 높은 온도]
scaled_probas = [
    softmax_with_temperature(next_token_logits, T)
    for T in temperatures
]

x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(6, 3))
for i, T in enumerate(temperatures):
    ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f"Temperature = {T}")

ax.set_ylabel("Probability")
ax.set_xticks(x + bar_width)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

### 5.3.2. Top-k 샘플링

* temperature를 높이는 방법은 텍스트 생성과정의 창의성을 높이지만 말이 안 되는 텍스트를 생성한다는 문제가 생김

* Top-k 샘플링 = 확률적 샘플링 + 온도 스케일링
    - 샘플링되는 토큰을 가장 가능성 있는 상위 k개 토큰으로 제한
    - 선택되지 않은 모든 로짓을 -inf 값으로 바꿈
    - 소프트맥스 값 계산시 top-k가 아닌 토큰의 확률 점수는 0이 되고 남은 토큰의 확률 합이 1이 됨

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
print("탑-k 로짓:", top_logits)
print("탑-k 위치:", top_pos)

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float('-inf')),
    other=next_token_logits
)

print(new_logits)

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

### 5.3.3. 텍스트 생성 함수 수정

#### 온도 스케일링 + 확률적 샘플링을 위한 multinomial 함수를 적용하여 0이 아닌 3개의 확률 점수에서 하나를 선택하여 다음 토큰 생성 가능

#### 다양성 증가를 위한 수정된 텍스트 생성 함수 (temperature scaling + top-k sampling 적용)

In [ ]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):

    # for 루프는 이전과 동일. 로짓을 받아 마지막 타임 스텝만 사용
    for _ in range(max_new_tokens):

        # 현재 context_size만큼의 토큰만 모델 입력으로 사용
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        # 마지막 타임스텝의 logits만 사용
        logits = logits[:, -1, :]

        # top-k 샘플링으로 로짓 필터링
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]

            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        # temperature sampling 적용
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # temperature가 0이면 greedy decoding
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # EOS 토큰을 만나면 생성 중단
        if idx_next == eos_id:
            break

        # 새 토큰을 기존 idx 뒤에 붙이기
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
torch.manual_seed(123)
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("출력 텍스트: \n", token_ids_to_text(token_ids, tokenizer))

## 5.4. pytorch로 모델 로드 + 저장

* 사전 훈련된 모델 저장 + 로드하는 방법

In [ ]:
# model.pth는 state_dit를 저장할 파일 이름
# 관례적으로 파이토치에서는 .pth 확장자를 사용하지만 기술적으로는 어떤 확장자도 사용 가능
torch.save(model.state_dict(), "model.pth")

#### state_dict로 모델 가중치 저장 후 이 가중치를 새로운 GPTModel 객체에 로드 가능함

In [ ]:
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()

#### model.eval() -> 모델을 추론을 위한 평가 모드로 전환하면 모델의 dropout층 비활성화 가능

#### 앞에서 정의한 train_model_simple 함수 사용하여 모델 훈련할 예정이라면 옵티마이저 상태도 저장하는 게 좋음

#### Adam W (적응형 옵티마이저)는 모델 가중치마다 추가 파라미터를 저장함
* 과거 데이터를 사용하여 각 모델 파라미터의 학습률을 동적으로 조정함
    - 이 데이터가 없으면 옵티마이저가 초기화되고 모델이 최적점이 아닌 곳으로 수렴 or 수렴 자체 실패
    (일관된 텍스트 생성 능력을 얻지 못함)

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)

#### 저장된 데이터를 먼저 torch.load로 로그 + load_state_dict 메서드로 모델과 옵티마이저 상태 복원

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", map_location=device)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train()

## 5.5. 오픈 AI에서 사전 훈련된 GPT-2 모델 가중치 로드
* 가중치: 파이토치의 Linear와 임베딩 층의 .weight 속성에 저장된 가중치 파라미터

#### 이 책 깃허브에서 직접 gpt_download.py 파이썬 모듈 다운로드

In [ ]:
import urllib.request

url = (
    "https://raw.githubusercontent.com/rickiepark/"
    "llm-from-scratch/main/ch05/"
    "01_main-chapter-code/gpt_download.py"
)

filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

#### gpt_download.py 파일에서 download_and_load_gpt2 함수 임포트
* GPT-2 구조 설정(settings)과 가중치 파라미터(params)를 현재 파이썬 세션에 로드

In [ ]:
import sys
print(sys.version)

In [ ]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(
    model_size="124M", models_dir="gpt2"
)

In [ ]:
print("설정: ", settings)
print("파라미터 딕셔너리 키: ", params.keys())

In [ ]:
print("GPT-2 settings:")
print(settings)

print("\nparams keys:")
print(params.keys())

print("토큰 임베딩 가중치 shape:", params["wte"].shape)
print("위치 임베딩 가중치 shape:", params["wpe"].shape)

#### settings, params: 모두 딕셔너리
* settings: 우리가 수동으로 만든 GPT_CONFIG_124M처럼 LLM 구조 설정 저장함
* params: 실제 가중치 텐서

#### 토큰 임베딩 층의 가중치

In [ ]:
print(params["wte"])
print("토큰 임베딩 가중치 텐서의 차원:", params["wte"].shape)

#### GPT-2 모델 가중치를 파이썬으로 로드한 후 settings와 params 딕셔너리를 GPTModel 객체로 복사
* 가장 작은 모델인 "gpt2-small (124M)"를 로드한다고 가정
* 앞서 정의한 GPT_CONFIG_124M 설정을 model_configs의 값으로 업데이트

In [ ]:
model_configs = {
    "gpt2-small (124M)": {"emb_dim":768, "n_layers":12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim":1024, "n_layers":24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim":1280, "n_layers":36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim":1600, "n_layers":48, "n_heads": 25},
}

In [ ]:
model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

#### 예제에서는 256 토큰 길이 사용
* 오픈 AI의 원본 GPT-2 모델: 1,024 토큰 길이로 훈련
    - 이를 반영하도록 NEW_CONFIG 업데이트 필요

In [ ]:
NEW_CONFIG.update({"context_length": 1024})

#### 오픈 AI에서는 멀티 헤드 어텐션 모듈에서 쿼리, 키, 값 행렬을 계산하는 선형 층에 편향 벡터 사용
* 편향 벡터: 모델 성능 향상에는 불필요 
* 따라서 LLM에서 더 이상 잘 사용되지 않음
* 사전 훈련된 가중치를 사용해야 하므로 편향 벡터를 사용할 수 있도록 설정 맞추기

In [ ]:
NEW_CONFIG.update({"qkv_bias": True})

#### NEW_CONFIG 딕셔너리를 통한 GPTModel 클래스 객체 초기화

In [ ]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval()

#### 오픈 AI 모델 가중치를 사용하기 위한 마지막 단계
* GPTModel의 객체가 랜덤한 가중치로 초기화되는데, 이 가중치를 params 딕셔너리에 있는 가중치로 바꾸기

#### assign 유틸리티 함수 정의
* 두 텐서 or 배열의 크기가 같은지 검사 + right를 훈련 가능한 파이토치 파라미터 텐서로 바꾸어 반환

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"크기가 다릅니다. left: {left.shape}, right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

#### params 딕셔너리의 가중치를 GPTModel의 인스턴스인 gpt로 로드하는 load_weights_into_gpt 함수 정의
* 오픈 AI 가중치를 GPTModel의 객체로 로드하기

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params): # 위치 임베딩과 토큰 임베딩의 가중치를 params의 값으로 설정
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])): # 모델의 트랜스포커 블록 순회
        q_w, k_w, v_w = np.split( # np.split 함수로 어텐션 가중치와 편향 가중치를 쿼리, 키, 값 세 부분으로 나눔
            params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T
        )
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T
        )
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T
        )

        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b
        )
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b
        )
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b
        )

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"]
        )

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"]
        )
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"]
        )

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"]
        )
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"]
        )
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"]
        )
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"]
        )

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    # 오픈 AI의 원본 GPT-2 모델은 토큰 임베딩의 가중치를 출력 층에 재사용하여 전체 파라미터 개수를 절감하는 가중치 묶기를 사용함
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

#### load_weights_into_gpt 함수 실행 + 오픈AI의 모델 가중치를 GPTModel의 인스턴스인 gpt로 로드

In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
torch.manual_seed(123)
token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

In [ ]:
print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))